In [32]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
import  os, urllib.request
import time

In [33]:
# Load models
vehicle_model = YOLO("yolov8n.pt")
if not os.path.exists("yolov8n-license-plate.pt"):
    urllib.request.urlretrieve(
        "https://huggingface.co/Koushim/yolov8-license-plate-detection/resolve/main/best.pt",
        "yolov8n-license-plate.pt"
    )
plate_model = YOLO("yolov8n-license-plate.pt")
reader = easyocr.Reader(['en'], gpu=False)


Using CPU. Note: This module is much faster with a GPU.


In [34]:
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    if inter == 0:
        return 0
    areaA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter)


In [35]:
cap = cv2.VideoCapture("video.mp4")
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

VEHICLE_CLASSES = [2, 3, 5, 7]
tracks = {}
next_id = 0
frame_id = 0

PLATE_INTERVAL = 8    # detect plate mỗi 8 frame
DISPLAY_SCALE = 0.8


In [36]:
last_time = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1

    # ---------- YOLO XE ----------
    results = vehicle_model(frame, imgsz=960, conf=0.4)[0]
    detections = []

    for box, cls, conf in zip(
        results.boxes.xyxy,
        results.boxes.cls,
        results.boxes.conf
    ):
        if int(cls) in VEHICLE_CLASSES and conf > 0.4:
            detections.append(tuple(map(int, box)))

    # ---------- TRACKING ----------
    new_tracks = {}

    for det in detections:
        best_iou, best_id = 0, None
        for tid, tr in tracks.items():
            val = iou(det, tr["box"])
            if val > best_iou:
                best_iou, best_id = val, tid

        if best_iou > 0.3:
            new_tracks[best_id] = tracks[best_id]
            new_tracks[best_id]["box"] = det
            new_tracks[best_id]["last_seen"] = frame_id
        else:
            new_tracks[next_id] = {
                "box": det,
                "plate_box": None,
                "last_seen": frame_id
            }
            next_id += 1

    tracks = new_tracks

    # ---------- YOLO PLATE (THƯA) ----------
    if frame_id % PLATE_INTERVAL == 0:
        for tid, tr in tracks.items():
            x1,y1,x2,y2 = tr["box"]
            vehicle_crop = frame[y1:y2, x1:x2]
            if vehicle_crop.size == 0:
                continue

            plates = plate_model(vehicle_crop, conf=0.25)[0]

            for pbox in plates.boxes.xyxy:
                px1,py1,px2,py2 = map(int, pbox)
                tr["plate_box"] = (px1,py1,px2,py2)
                break   # 1 plate / xe

    # ---------- VẼ ----------
    for tid, tr in tracks.items():
        x1,y1,x2,y2 = tr["box"]
        cv2.rectangle(frame,(x1,y1),(x2,y2),(255,0,0),2)
        cv2.putText(frame,f"ID {tid}",
                    (x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,(255,0,0),2)

        if tr["plate_box"] is not None:
            px1,py1,px2,py2 = tr["plate_box"]
            cv2.rectangle(frame,
                (x1+px1, y1+py1),
                (x1+px2, y1+py2),
                (0,255,0),2)

    # ---------- FPS ----------
    now = time.time()
    fps = 1/(now-last_time)
    last_time = now
    cv2.putText(frame,f"FPS: {int(fps)}",(20,40),
                cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

    frame = cv2.resize(frame,None,fx=DISPLAY_SCALE,fy=DISPLAY_SCALE)
    cv2.imshow("YOLO Plate Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()



0: 544x960 9 persons, 18 cars, 7 motorcycles, 1 bus, 3 trucks, 142.4ms
Speed: 16.9ms preprocess, 142.4ms inference, 1.7ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 7 persons, 18 cars, 8 motorcycles, 2 buss, 2 trucks, 375.7ms
Speed: 10.4ms preprocess, 375.7ms inference, 2.8ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 8 persons, 17 cars, 8 motorcycles, 3 buss, 1 truck, 572.0ms
Speed: 10.2ms preprocess, 572.0ms inference, 2.0ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 6 persons, 18 cars, 8 motorcycles, 2 buss, 3 trucks, 789.5ms
Speed: 15.9ms preprocess, 789.5ms inference, 4.2ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 5 persons, 18 cars, 8 motorcycles, 2 buss, 3 trucks, 473.3ms
Speed: 9.5ms preprocess, 473.3ms inference, 1.2ms postprocess per image at shape (1, 3, 544, 960)

0: 544x960 6 persons, 19 cars, 10 motorcycles, 1 bus, 1 truck, 112.4ms
Speed: 5.9ms preprocess, 112.4ms inference, 1.4ms postprocess per image

KeyboardInterrupt: 

In [37]:
cap.release()
cv2.destroyAllWindows()